# Experiment 029 (W14b): GRPO after MC-SFT at 1.5B and 3B (Colab)

Six arms, seeds 0 to 2 at each of the two smaller scales, each starting grpo
from its seed-matched p8 mc-sft adapter (base to SFT(MMLU) to GRPO(MMLU)),
1.5B from the 010 parents and 3B from the 007 parents. The registered rules
from the W14b plan apply: adapter inventory preflight with hash bindings
before any contact, parent identity probes in disposable subprocesses, the
pre-training pool variability diagnostic at zero test-set contact, per-scale
GPU gates and a 40GB Drive gate, base evals prepopulated from 009 and 003 and
never recomputed, training through the disposable worker with the merge bound
to its source, and the one registered loop change: an exact a2 sentinel abort
concludes its arm and the loop continues, every other failure still halts.

One Run-all per session; every cell is resume safe.

## 0. Compute units and mount Drive

In [ ]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before =  671.94     # fill from the resources panel
unit_rate = 6.77     # units per hour for this card
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '029_colab_mcsft_grpo_small',
                 'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(start, f)
        print('opened session record')

session 20260812_082351 | cu_before = 671.94
Mounted at /content/drive
opened session record


## 1. GPU check: A100 with 80GB required

In [ ]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'total vram: {total_gb:.1f} GiB')
assert 'A100' in gpu and total_gb >= 79, 'need the a100 80gb runtime, the card both scale cells ran on per rule 6z'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-1e1c2cc5-6f4a-8f2a-3a31-4396c8082134)
total vram: 79.3 GiB


## 2. Clone or update the repo (automatic)

In [ ]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 3690, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 3690 (delta 5), reused 8 (delta 5), pack-reused 3674 (from 1)
Receiving objects: 100% (3690/3690), 111.89 MiB | 34.94 MiB/s, done.
Resolving deltas: 100% (2404/2404), done.
Updating files: 100% (1684/1684), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [ ]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 159.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.9 MB/s eta 0:00:00


## 4. W&B key (automatic)

In [ ]:
import getpass, os
key = None
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
except Exception:
    key = None
if not key and os.path.exists('wandb_api_key.txt'):
    key = open('wandb_api_key.txt').read().strip()
if not key:
    key = getpass.getpass('wandb api key: ').strip()
assert key, 'a wandb key is required, the worker logs online and must not wait for a login'
os.environ['WANDB_API_KEY'] = key
with open('wandb_api_key.txt', 'w') as f:
    f.write(key)
key = None
print('wandb key set')

wandb key set


## 5. Arm selection: all six arms are registered for the first session

In [ ]:
# the staging switch from 028 carries over: narrowing this list stages a
# session without touching the protocol, and a rerun skips whatever finished
# or concluded. All six arms are selected per the registered plan
ARMS_TO_RUN = [('B15', 0), ('B15', 1), ('B15', 2), ('B3', 0), ('B3', 1), ('B3', 2)]
print('arms this session:', ARMS_TO_RUN)


arms this session: [('B15', 0), ('B15', 1), ('B15', 2), ('B3', 0), ('B3', 1), ('B3', 2)]


## 6. Preflight: source adapters, parents, capacity, before any contact

In [ ]:
import json, os, shutil, sys
sys.path.insert(0, os.getcwd())
from config import load_config, run_tag
from scripts.s12_eval import atomic_write_json, config_hash_status
from scripts.s28_compose import ARMS_029, SCALES_029, resolve_source_adapter

D = '/content/drive/MyDrive/inm363'
# two parent generations, so the acceptable hash era is scale-exact: the 010
# parents predate only the grpo chain field, the 007 parents predate both
ERA_OK = {'B15': ('current', 'legacy_missing_grpo_from_adapter'),
          'B3': ('current', 'legacy_missing_both_chain_fields')}

def scale_of(cfg_path):
    return 'B15' if '_b15_' in cfg_path else 'B3'

ADIR, PREFLIGHT, TAGS = {}, {}, {}
problems = []
for cfg_path, seed, parent_tag in ARMS_029:
    scale = scale_of(cfg_path)
    key = (scale, seed)
    cfg = load_config(cfg_path)
    assert cfg.train.seed == seed and cfg.scale.name == scale
    TAGS[key] = run_tag(cfg)
    try:
        adir, hashes = resolve_source_adapter(cfg.train.grpo_from_adapter, D)
    except Exception as e:
        problems.append(f'{key}: {e}')
        continue
    ADIR[key], PREFLIGHT[key] = adir, hashes

    # the stored hash must belong to this exact parent, current or its era
    parent_cfg = load_config(SCALES_029[scale]['parent_config'])
    parent_cfg.train.seed = seed
    parent_man = json.load(open(os.path.join(D, 'results', parent_tag, 'manifest.json')))
    stored = open(os.path.join(adir, 'config_hash.txt')).read().strip()
    status = config_hash_status(parent_cfg, parent_man, stored)
    if status not in ERA_OK[scale]:
        problems.append(f'{key}: adapter hash status {status}')
        continue

    # the parent artifacts the gates and the summary reference need
    pdir = os.path.join(D, 'results', parent_tag)
    for n in ('sft_logprob_preds.jsonl', 'sft_logprob.json', 'sft_eval.json', 'sft_preds.jsonl'):
        if not os.path.exists(os.path.join(pdir, n)):
            problems.append(f'{key}: parent missing {n}')

    # the q2 pipeline arm must exist with a manifest for the env assert
    pipe_tag = SCALES_029[scale]['pipeline_fmt'].format(s=seed)
    if not os.path.exists(os.path.join(D, 'results', pipe_tag, 'manifest.json')):
        problems.append(f'{key}: pipeline manifest missing for {pipe_tag}')
    print(f'ok: {key} <- {parent_tag} ({status})')

free_gb = shutil.disk_usage(D).free / 1024**3
print(f'drive free: {free_gb:.0f} GB')
if free_gb < 40:
    problems.append(f'only {free_gb:.0f}GB free on drive, six merged bases plus '
                    'checkpoints need 40')

assert not problems, 'preflight halt:\n' + '\n'.join(problems)

# the hashes persist on drive; a rerun must see the same adapters, not trust memory
pf_path = os.path.join(D, 'results', '029_preflight.json')
record = {TAGS[k]: PREFLIGHT[k] for k in PREFLIGHT}
if os.path.exists(pf_path):
    prev = json.load(open(pf_path))
    assert prev == record, 'source adapters changed since the recorded preflight, halting'
    print('preflight matches the recorded hashes')
else:
    atomic_write_json(pf_path, record)
    print('preflight recorded')
print('preflight green for all six arms')


ok: ('B15', 0) <- 010_p8_COLAB_FULL_B15_sft_only_seed0 (legacy_missing_grpo_from_adapter)
ok: ('B15', 1) <- 010_p8_COLAB_FULL_B15_sft_only_seed1 (legacy_missing_grpo_from_adapter)
ok: ('B15', 2) <- 010_p8_COLAB_FULL_B15_sft_only_seed2 (legacy_missing_grpo_from_adapter)
ok: ('B3', 0) <- 007_p8_COLAB_FULL_B3_sft_only_seed0 (legacy_missing_both_chain_fields)
ok: ('B3', 1) <- 007_p8_COLAB_FULL_B3_sft_only_seed1 (legacy_missing_both_chain_fields)
ok: ('B3', 2) <- 007_p8_COLAB_FULL_B3_sft_only_seed2 (legacy_missing_both_chain_fields)
drive free: 178 GB
preflight recorded
preflight green for all six arms


## 7. Build the data splits (after the preflight)

In [ ]:
!python scripts/build_data.py

README.md: 100% 7.93k/7.93k [00:00<00:00, 22.1MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:00<00:00, 2.57MB/s,  226kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:00<00:00, 2.57MB/s,  227kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:00<00:00, 710kB/s, 41.5kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:00<00:00, 710kB/s, 41.5kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 393630.81 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 336002.85 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 63.9MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 146MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B

## 8. Base evals: prepopulated from 009 and 003, never recomputed

In [ ]:
from scripts.s28_compose import prepopulate_base_evals
for scale, exp in (('B15', '009'), ('B3', '003')):
    src = os.path.join(D, 'results', 'base_evals', f'{exp}_COLAB_FULL_{scale}')
    dst = os.path.join(D, 'results', 'base_evals', f'029_COLAB_FULL_{scale}')
    rec = prepopulate_base_evals(src, dst)
    print(scale, 'reuse record covers', sorted(rec['sha256']))


  base evals prepopulated from 009_COLAB_FULL_B15
B15 reuse record covers ['base_eval.json', 'base_logprob.json', 'base_logprob_preds.jsonl', 'base_preds.jsonl']
  base evals prepopulated from 003_COLAB_FULL_B3
B3 reuse record covers ['base_eval.json', 'base_logprob.json', 'base_logprob_preds.jsonl', 'base_preds.jsonl']


## 9. Wiring check: the package tests run inside the session

In [ ]:
# the package tests run inside the session and a failure stops everything here
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest',
                    'tests/test_s28_composition.py', 'tests/test_checkpoint.py', '-q'])
assert r.returncode == 0, 'package tests failed, nothing runs'

from scripts.grpo_worker import (active_worker_processes, blocked_resume_reason,
                                 stream_subprocess)
from scripts.s12_eval import GateFailure, IntegrityError, identity_check, read_jsonl, rows_aligned, sha256
from scripts.s28_compose import (arm_state, commit_endpoint_eval, commit_pool_variability,
                                 conclude_a2, concluded_a2_state, endpoint_state,
                                 gate_state, pool_variability_state,
                                 sentinel_abort_reason, status_is_a2, verify_binding,
                                 write_arm_complete)
from obs import _versions, append_ledger, gpu_snapshot, _git_info
from train.model_utils import resolve_dtype
print('imports fine | tags:', [TAGS[s] for s in TAGS])

imports fine | tags: ['029_COLAB_FULL_B15_mcsft_grpo_seed0', '029_COLAB_FULL_B15_mcsft_grpo_seed1', '029_COLAB_FULL_B15_mcsft_grpo_seed2', '029_COLAB_FULL_B3_mcsft_grpo_seed0', '029_COLAB_FULL_B3_mcsft_grpo_seed1', '029_COLAB_FULL_B3_mcsft_grpo_seed2']


In [ ]:
!pip uninstall -q -y torchao

## 10. Parent identity probes, each in a disposable subprocess

In [ ]:
# all six parent identity probes run before any training, whatever the arm
# selection, so the probe contact stays fixed at six passes. Each probe runs
# in a child process that dies and releases its memory, and each completed gate
# persists on drive so a runtime restart never repeats the contact
import shutil as _shutil, subprocess, sys, time
from scripts.s12_eval import atomic_write_json

GATE_SCRIPT = '/content/s29_gate.py'
with open(GATE_SCRIPT, 'w') as f:
    f.write('''import argparse, os, sys
sys.path.insert(0, os.getcwd())
from config import load_config
from scripts.env_switch import apply_smoke, resolve_device
from eval.harness import run_logprob_eval

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--adapter", required=True)
ap.add_argument("--out", required=True)
args = ap.parse_args()
cfg = apply_smoke(load_config(args.config))
run_logprob_eval(cfg, resolve_device(cfg.env), adapter_path=args.adapter,
                 condition="gate", out_dir=args.out)
''')

for cfg_path, seed, parent_tag in ARMS_029:
    key = (scale_of(cfg_path), seed)
    gate_path = os.path.join(D, 'results', TAGS[key], 'parent_gate.json')
    os.makedirs(os.path.dirname(gate_path), exist_ok=True)
    archived_p = os.path.join(D, 'results', parent_tag, 'sft_logprob_preds.jsonl')
    if gate_state(gate_path, PREFLIGHT[key], archived_p) == 'committed':
        print(f'{key}: gate already committed, skipping')
        continue

    gate_dir = f'/content/s29_gate_{key[0]}_seed{seed}'
    gate_preds = os.path.join(gate_dir, 'gate_logprob_preds.jsonl')
    # an incomplete scratch from a dead runtime is rebuilt, never trusted
    rows_ok = (os.path.exists(gate_preds)
               and sum(1 for l in open(gate_preds) if l.strip()) == 1884)
    if not rows_ok:
        _shutil.rmtree(gate_dir, ignore_errors=True)
        t0 = time.time()
        r = subprocess.run([sys.executable, GATE_SCRIPT, '--config', cfg_path,
                            '--adapter', ADIR[key], '--out', gate_dir])
        assert r.returncode == 0, f'{key}: gate probe subprocess failed'
        elapsed = round(time.time() - t0, 1)
    else:
        elapsed = None
    new_rows = read_jsonl(gate_preds)
    assert len(new_rows) == 1884, f'{key}: gate scratch is short'
    stats = identity_check(new_rows, read_jsonl(archived_p))
    atomic_write_json(gate_path, {
        'ts': time.strftime('%Y-%m-%d %H:%M'), 'completed': True,
        'parent': parent_tag, 'adapter_sha256': PREFLIGHT[key],
        'archived_preds_sha256': sha256(archived_p),
        'new_preds_sha256': sha256(gate_preds),
        'stats': stats, 'elapsed_sec': elapsed})
    _shutil.rmtree(gate_dir, ignore_errors=True)
    print(f'{key} gate: {stats}')

import torch
free_gib = torch.cuda.mem_get_info()[0] / 1024**3
print(f'gpu free: {free_gib:.1f} GiB')
# the training bar is per scale and asserted inside the run loop

('B15', 0) gate: {'agreement': 1.0, 'acc_new': 0.693206, 'acc_archived': 0.693206}
('B15', 1) gate: {'agreement': 1.0, 'acc_new': 0.697983, 'acc_archived': 0.697983}
('B15', 2) gate: {'agreement': 1.0, 'acc_new': 0.695329, 'acc_archived': 0.695329}
('B3', 0) gate: {'agreement': 1.0, 'acc_new': 0.747877, 'acc_archived': 0.747877}
('B3', 1) gate: {'agreement': 1.0, 'acc_new': 0.747877, 'acc_archived': 0.747877}
('B3', 2) gate: {'agreement': 1.0, 'acc_new': 0.746815, 'acc_archived': 0.746815}
gpu free: 78.8 GiB


## 11. Pool variability diagnostic: the fuel measure, before any training

In [ ]:
# samples every parent on the 245-prompt training pool at the frozen grpo
# sampling settings and records how much within-group variety grpo will get.
# training pool only, so zero test-set contact. Record only, no decision weight
import shutil as _shutil, subprocess, sys, time

DIAG_SCRIPT = '/content/s29_diag.py'
with open(DIAG_SCRIPT, 'w') as f:
    f.write('''import argparse, json, os, sys
sys.path.insert(0, os.getcwd())
import torch
from config import load_config
from data.mmlu import extract_letter
from data.splits import build_grpo_split
from data.tagging import is_format_ok, with_open_tag
from scripts.env_switch import apply_smoke, resolve_device
from train.model_utils import load_model_and_tokenizer, set_all_seeds

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--adapter", required=True)
ap.add_argument("--out", required=True)
args = ap.parse_args()
cfg = apply_smoke(load_config(args.config))
device = resolve_device(cfg.env)
set_all_seeds(cfg.train.seed)
rows = build_grpo_split()
model, tok = load_model_and_tokenizer(cfg, device, adapter_path=args.adapter)
model.eval()

K = cfg.train.num_generations
BATCH = 8
stats = []
for i in range(0, len(rows), BATCH):
    chunk = rows[i:i + BATCH]
    enc = tok([r["prompt"] for r in chunk], return_tensors="pt", padding=True,
              padding_side="left").to(device)
    with torch.no_grad():
        out = model.generate(**enc, do_sample=True, temperature=cfg.grpo.temperature,
                             top_p=cfg.grpo.top_p, num_return_sequences=K,
                             max_new_tokens=cfg.grpo.max_completion_length,
                             pad_token_id=tok.pad_token_id)
    gen = out[:, enc["input_ids"].shape[1]:]
    texts = tok.batch_decode(gen, skip_special_tokens=True)
    for j in range(len(chunk)):
        group = texts[j * K:(j + 1) * K]
        toks = gen[j * K:(j + 1) * K]
        # pad-token counting also drops the eos, close enough for a diagnostic
        lens = [(t != tok.pad_token_id).sum().item() for t in toks]
        gold = chunk[j]["gold"]
        answers = [extract_letter(with_open_tag(c)) for c in group]
        # the exact reward the trainer would hand grpo, per completion
        accs = [cfg.reward.correct_weight if a == gold else 0.0 for a in answers]
        fmts = [cfg.reward.format_weight
                if is_format_ok(with_open_tag(c), cfg.reward.min_think_chars) else 0.0
                for c in group]
        rewards = [a + f for a, f in zip(accs, fmts)]
        mean_r = sum(rewards) / K
        var = sum((r - mean_r) ** 2 for r in rewards) / K
        stats.append({"d": len(set(group)), "da": len(set(answers)),
                      "t": sum(lens) / K,
                      "acc": sum(1 for a in accs if a > 0) / K,
                      "fmt": sum(1 for f in fmts if f > 0) / K,
                      "rstd": var ** 0.5})
n = len(stats)
record = {"n_prompts": n, "samples_per_prompt": K,
          "temperature": cfg.grpo.temperature,
          "share_single_completion": sum(1 for s in stats if s["d"] == 1) / n,
          "mean_distinct_completions": sum(s["d"] for s in stats) / n,
          "share_single_answer": sum(1 for s in stats if s["da"] == 1) / n,
          "mean_completion_tokens": sum(s["t"] for s in stats) / n,
          "mean_accuracy_reward": sum(s["acc"] for s in stats) / n,
          "format_rate": sum(s["fmt"] for s in stats) / n,
          "share_zero_reward_spread": sum(1 for s in stats if s["rstd"] == 0.0) / n,
          "mean_reward_std": sum(s["rstd"] for s in stats) / n}
with open(args.out, "w") as f:
    json.dump(record, f)
print("pool variability:", json.dumps(record))
''')

for cfg_path, seed, parent_tag in ARMS_029:
    key = (scale_of(cfg_path), seed)
    res_dir = os.path.join(D, 'results', TAGS[key])
    os.makedirs(res_dir, exist_ok=True)
    if pool_variability_state(res_dir, PREFLIGHT[key]) == 'committed':
        print(f'{key}: diagnostic already committed, skipping')
        continue
    tmp_out = f'/content/s29_diag_{key[0]}_seed{seed}.json'
    if os.path.exists(tmp_out):
        os.remove(tmp_out)
    t0 = time.time()
    r = subprocess.run([sys.executable, DIAG_SCRIPT, '--config', cfg_path,
                        '--adapter', ADIR[key], '--out', tmp_out])
    assert r.returncode == 0, f'{key}: diagnostic subprocess failed'
    rec = json.load(open(tmp_out))
    rec.update({'parent_tag': parent_tag, 'adapter_sha256': PREFLIGHT[key],
                'split_revision': _versions().get('cais/mmlu@', ''),
                'ts': time.strftime('%Y-%m-%d %H:%M'),
                'elapsed_sec': round(time.time() - t0, 1)})
    commit_pool_variability(res_dir, rec)
    os.remove(tmp_out)
    print(f"{key}: single-completion groups {rec['share_single_completion']:.1%}, "
          f"zero-spread groups {rec['share_zero_reward_spread']:.1%}, "
          f"pool accuracy {rec['mean_accuracy_reward']:.3f}, "
          f"mean tokens {rec['mean_completion_tokens']:.1f}")


('B15', 0): single-completion groups 17.6%, zero-spread groups 83.3%, pool accuracy 0.993, mean tokens 16.0
('B15', 1): single-completion groups 26.9%, zero-spread groups 95.5%, pool accuracy 0.990, mean tokens 16.3
('B15', 2): single-completion groups 40.8%, zero-spread groups 96.7%, pool accuracy 0.995, mean tokens 16.7
('B3', 0): single-completion groups 73.5%, zero-spread groups 88.6%, pool accuracy 0.987, mean tokens 16.0
('B3', 1): single-completion groups 25.3%, zero-spread groups 71.0%, pool accuracy 0.996, mean tokens 15.9
('B3', 2): single-completion groups 80.0%, zero-spread groups 91.4%, pool accuracy 0.987, mean tokens 16.0


## 12. The run: bound merge, worker training, abort-continue, standard evaluation

In [ ]:
# both cost fields must be filled before the long runs start
assert cu_before is not None and unit_rate is not None, 'fill cu_before and unit_rate in the units cell'
import glob as _glob, subprocess, sys, time
from datetime import datetime

EVAL_SCRIPT = '/content/s29_eval.py'
with open(EVAL_SCRIPT, 'w') as f:
    f.write('''import argparse, os, sys
sys.path.insert(0, os.getcwd())
from checkpoint import save_metrics
from config import load_config
from scripts.env_switch import apply_smoke, resolve_device
from eval.harness import run_eval, run_logprob_eval

ap = argparse.ArgumentParser()
ap.add_argument("--config", required=True)
ap.add_argument("--adapter", required=True)
ap.add_argument("--out", required=True)
args = ap.parse_args()
cfg = apply_smoke(load_config(args.config))
device = resolve_device(cfg.env)
ev = run_eval(cfg, device, adapter_path=args.adapter, condition="grpo", out_dir=args.out)
save_metrics(os.path.join(args.out, "grpo_eval.json"), ev)
lp = run_logprob_eval(cfg, device, adapter_path=args.adapter, condition="grpo", out_dir=args.out)
save_metrics(os.path.join(args.out, "grpo_logprob.json"), lp)
''')

os.environ['WANDB_PROJECT'] = 'inm363-grpo'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
LEDGER = os.path.join(D, 'results', 'compute_ledger.jsonl')
allowed_infra_files = {
    'notebooks/029_colab_mcsft_grpo_small.ipynb',
    'scripts/grpo_worker.py',
    'tests/test_s28_composition.py',
    'tests/test_grpo_worker.py',
}

for cfg_path, seed, parent_tag in ARMS_029:
    scale = scale_of(cfg_path)
    key = (scale, seed)
    if key not in ARMS_TO_RUN:
        continue
    tag = TAGS[key]
    print('===== ' + tag + ' =====')
    cfg = load_config(cfg_path)

    # each arm checks against the environments both parents recorded, dataset
    # revision pins included where the manifest carries them
    for man_tag in (parent_tag, SCALES_029[scale]['pipeline_fmt'].format(s=seed)):
        parent = json.load(open(os.path.join(D, 'results', man_tag, 'manifest.json')))
        for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate',
                    'cais/mmlu@', 'openai/gsm8k@'):
            want = parent.get('versions', {}).get(lib)
            if want is not None:
                got = _versions().get(lib)
                assert got == want, f'{man_tag} {lib}: {got} vs {want}'
        if parent.get('resolved_dtype'):
            assert resolve_dtype(cfg) == parent['resolved_dtype'], f'{man_tag}: dtype differs'
    assert cfg.exec.allow_oom_fallback is False, 'the ladder must stay off'
    assert cfg.log.wandb_project == 'inm363-grpo', 'wandb project comes from the config'

    out_dir = os.path.join(D, 'checkpoints', tag, 'grpo')
    merged_dir = os.path.join(D, 'checkpoints', tag, 'sft_merged')
    res_dir = os.path.join(D, 'results', tag)
    os.makedirs(res_dir, exist_ok=True)
    final_artifacts = [os.path.join(out_dir, 'adapter_model.safetensors'),
                       os.path.join(out_dir, 'adapter_base.json'),
                       os.path.join(res_dir, 'grpo_trainlog.json')]

    # fully done only when the completion record verifies the summary, the
    # manifest and the adapter-bound endpoint; anything inconsistent halts
    if arm_state(res_dir, out_dir) == 'committed':
        verify_binding(ADIR[key], merged_dir)
        print('  arm complete, skipping')
        continue

    man_path = os.path.join(res_dir, 'arm_manifest.json')
    man = json.load(open(man_path)) if os.path.exists(man_path) else {'attempts': []}

    def newest_step():
        cks = _glob.glob(os.path.join(out_dir, 'checkpoint-*'))
        return max((int(c.rsplit('-', 1)[1]) for c in cks), default=0)

    start_step = newest_step()
    for a in man.get('attempts', []):
        if 'ts_end' not in a:
            a.update({'interrupted': True, 'end_step': start_step,
                      'elapsed_sec': None, 'ts_end': 'runtime died before finally'})
    with open(man_path, 'w') as f:
        json.dump(man, f, indent=2)

    # the conclusion record is examined on every pass through this arm, right
    # after the manifest loads: committed skips, any contradiction between the
    # record, the manifest and the results dir halts, absent falls through
    if concluded_a2_state(res_dir, man.get('attempts', [])) == 'committed':
        print('  concluded by sentinel a2 previously, skipping')
        continue
    a2_prior = sentinel_abort_reason(man.get('attempts', []))
    if a2_prior:
        # the manifest ends in a2 but the record is missing, a runtime died
        # between the two writes: repair from the validated diagnostic only
        assert pool_variability_state(res_dir, PREFLIGHT[key]) == 'committed', \
            f'{key}: diagnostic record missing, run the diagnostic cell first'
        pv = json.load(open(os.path.join(res_dir, 'pool_variability.json')))
        conclude_a2(res_dir, man['attempts'][-1], evidence=pv)
        print(f'  concluded by sentinel a2 previously, skipping: {a2_prior[:80]}')
        continue
    reason = blocked_resume_reason(man.get('attempts', []))
    assert reason is None, f'{key}: {reason}'
    assert not active_worker_processes(cfg_path), 'a training worker is already running'

    trained = all(os.path.exists(f) for f in final_artifacts)
    if not trained:
        # code drift between attempts is only tolerated for the named
        # infrastructure files, the 022 rule
        env = {'git': _git_info(), 'versions': _versions(),
               'resolved_dtype': resolve_dtype(cfg), 'ladder': 'disabled'}
        repo_top = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True,
                                  text=True, check=True).stdout.strip()
        repo_prefix = os.path.relpath(os.getcwd(), repo_top)
        for a in man.get('attempts', []):
            prev = a.get('env')
            if prev:
                old_c, new_c = prev['git'].get('commit'), env['git'].get('commit')
                if old_c != new_c:
                    diff = subprocess.run(['git', 'diff', '--name-only', old_c, new_c],
                                          capture_output=True, text=True, check=True).stdout.splitlines()
                    changed = []
                    for name in diff:
                        prefix = repo_prefix + os.sep
                        changed.append(name[len(prefix):] if repo_prefix != '.' and name.startswith(prefix) else name)
                    unexpected = sorted(set(changed) - allowed_infra_files)
                    assert not unexpected, f'seed {seed}: scientific code drifted between attempts: {unexpected}'
                for lib in ('torch', 'trl', 'transformers', 'peft', 'accelerate'):
                    assert prev['versions'].get(lib) == env['versions'].get(lib), f'seed {seed}: {lib} drifted'
                assert prev['resolved_dtype'] == env['resolved_dtype'], f'seed {seed}: dtype drifted between attempts'

        free_gib = torch.cuda.mem_get_info()[0] / 1024**3
        need_gib = SCALES_029[scale]['vram_gate_gib']
        assert free_gib >= need_gib, f'only {free_gib:.1f} GiB free, {scale} needs {need_gib}'

        # no training without the validated diagnostic, even when this cell
        # runs by itself: the conclusion evidence must never be empty
        assert pool_variability_state(res_dir, PREFLIGHT[key]) == 'committed', \
            f'{key}: diagnostic record missing, run the diagnostic cell first'
        pool_rec = json.load(open(os.path.join(res_dir, 'pool_variability.json')))

        attempt = {'start_step': start_step, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                   'gpu': gpu_snapshot(), 'env': env,
                   'adapter_sha256': PREFLIGHT[key], 'completed': False}
        man['attempts'].append(attempt)
        with open(man_path, 'w') as f:
            json.dump(man, f, indent=2)

        status_path = f'/content/029_worker_{key[0]}_seed{seed}.json'
        if os.path.exists(status_path):
            os.remove(status_path)
        worker_cmd = [sys.executable, '-u', 'scripts/grpo_worker.py',
                      '--config', cfg_path, '--adapter', ADIR[key],
                      '--out-dir', out_dir, '--log-dir', res_dir,
                      '--run-name', f'{tag}_grpo', '--status', status_path,
                      '--report-to', 'wandb']
        t0 = time.time()
        proc = None
        parent_failure = None
        a2_concluded = False
        try:
            print(f'===== {key}, from step {start_step} =====')
            proc = stream_subprocess(worker_cmd)
            if proc.returncode != 0:
                # the one registered exception: an exact a2 abort concludes the
                # arm and the session moves on. Anything else still halts here
                status_now = json.load(open(status_path)) if os.path.exists(status_path) else {}
                if status_is_a2(status_now):
                    a2_concluded = True
                else:
                    raise RuntimeError(f'{key}: training worker failed with exit code {proc.returncode}')
        except BaseException as exc:
            parent_failure = {'error_type': type(exc).__name__,
                              'error_message': str(exc)[:2000]}
            raise
        finally:
            wall = time.time() - t0
            end_step = newest_step()
            worker_returncode = proc.returncode if proc is not None else None
            worker_status = json.load(open(status_path)) if os.path.exists(status_path) else {}
            worker_peak = worker_status.get('peak_vram_gib')
            error_type = worker_status.get('error_type') or (parent_failure or {}).get('error_type')
            error_message = worker_status.get('error_message') or (parent_failure or {}).get('error_message')
            manual_interrupt = error_type == 'KeyboardInterrupt'
            explicit_failure = ((worker_returncode not in (None, 0) or parent_failure is not None)
                                and not manual_interrupt)
            done = (worker_status.get('completed') is True
                    and all(os.path.exists(f) for f in final_artifacts))
            attempt.update({'end_step': end_step, 'ts_end': time.strftime('%Y-%m-%d %H:%M'),
                            'elapsed_sec': round(wall, 1), 'completed': done,
                            'peak_vram_gib': worker_peak, 'worker_returncode': worker_returncode,
                            'error_type': error_type, 'error_message': error_message,
                            'cuda_oom': bool(worker_status.get('cuda_oom', False)),
                            'interrupted': manual_interrupt, 'resume_blocked': explicit_failure})
            with open(man_path, 'w') as f:
                json.dump(man, f, indent=2)
            gpu_row = gpu_snapshot()
            if worker_peak is not None:
                gpu_row['peak_vram_gb'] = worker_peak
            append_ledger(LEDGER, {'run_tag': tag, 'condition': 'MCSFT_GRPO_train',
                                   'timings_sec': {'grpo_train': round(wall, 1)},
                                   'train_steps': end_step - start_step,
                                   'completed': attempt['completed'], 'gpu': gpu_row})
        if a2_concluded:
            conclude_a2(res_dir, man['attempts'][-1], evidence=pool_rec)
            print(f'  concluded by sentinel a2 at the step-150 check, peak {worker_peak} GiB, next arm')
            continue
        assert worker_status.get('completed') is True, f'{key}: worker status did not complete'
        assert attempt['completed'] is True, f'{key}: final training artifacts are incomplete'
        print(f'  trained in {wall/3600:.2f}h, peak {worker_peak} GiB')
    else:
        print('  adapter already trained')

    # the trained-from weights re-verify before any evaluation, every time
    verify_binding(ADIR[key], merged_dir)
    _, hashes_now = resolve_source_adapter(cfg.train.grpo_from_adapter, D)
    assert hashes_now == PREFLIGHT[key], f'{key}: source adapter drifted since preflight'

    # Transactional endpoint evaluation: subprocess into scratch, then commit.
    # A tampered endpoint halts here, before any new test-set contact
    ep_state = endpoint_state(res_dir, out_dir)
    if ep_state == 'tampered':
        raise IntegrityError(f'{key}: endpoint files contradict their manifest, '
                             'halting before any re-evaluation')
    if ep_state != 'committed':
        sdir = f'/content/s29_eval_{key[0]}_seed{seed}'
        _shutil.rmtree(sdir, ignore_errors=True)
        t0 = time.time()
        eval_done = False
        try:
            r = subprocess.run([sys.executable, EVAL_SCRIPT, '--config', cfg_path,
                                '--adapter', out_dir, '--out', sdir])
            assert r.returncode == 0, f'{key}: evaluation subprocess failed'
            eval_wall = round(time.time() - t0, 1)
            gate_rec = json.load(open(os.path.join(res_dir, 'parent_gate.json')))
            extra = {'adapter': out_dir, 'source_adapter': ADIR[key],
                     'source_adapter_sha256': PREFLIGHT[key],
                     'merged_binding': verify_binding(ADIR[key], merged_dir),
                     'parent_gate_stats': gate_rec['stats'],
                     'config': cfg_path, 'seed': seed, 'git': _git_info(),
                     'versions': _versions(), 'resolved_dtype': resolve_dtype(cfg),
                     'gpu': gpu_snapshot(), 'timings_sec': {'grpo_eval': eval_wall}}
            commit_endpoint_eval(res_dir, sdir, adapter_dir=out_dir, extra_manifest=extra,
                                 package='029_w14b')
            _shutil.rmtree(sdir, ignore_errors=True)
            eval_done = True
        finally:
            append_ledger(LEDGER, {'run_tag': tag, 'condition': 'MCSFT_GRPO_eval',
                                   'timings_sec': {'grpo_eval': round(time.time() - t0, 1)},
                                   'train_steps': 0, 'completed': eval_done,
                                   'gpu': gpu_snapshot()})

    # summary and manifest through proven runner: training, base evals and
    # the committed endpoint files all skip inside it
    r = subprocess.run([sys.executable, 'scripts/run.py', '--config', cfg_path,
                        '--stage', 'all', '--no-ledger'])
    assert r.returncode == 0, f'{key}: summary stage failed'
    s = json.load(open(os.path.join(res_dir, 'summary.json')))
    assert s['reference'].startswith('mcsft_parent:'), 'summary must reference the mc-sft parent'
    ev = json.load(open(os.path.join(res_dir, 'grpo_eval.json')))
    write_arm_complete(res_dir)
    print('  gen per-tier:', ev['per_tier_acc'])

print('session done for arms', ARMS_TO_RUN)

Streaming output truncated to the last 5000 lines.
│ ├───────────────┼───────────────┼───────────────┼──────────────┼───────────┤ │
│ │ Answer the    │  I will       │          1.00 │         0.20 │      0.00 │ │
│ │ multiple      │ provide the   │               │              │           │ │
│ │ choice        │ selected      │               │              │           │ │
│ │ question.     │ option        │               │              │           │ │
│ │ Think through │ below.</thin… │               │              │           │ │
│ │ it step by    │ <answer>C</a… │               │              │           │ │
│ │ step inside   │               │               │              │           │ │
│ │ <think></thi… │               │               │              │           │ │
│ │ tags, then    │               │               │              │           │ │
│ │ give only the │               │               │              │           │ │
│ │ letter of the │               │               │       

## 13. Peek at the results

In [ ]:
for cfg_path, seed, parent_tag in ARMS_029:
    key = (scale_of(cfg_path), seed)
    res_dir = os.path.join(D, 'results', TAGS[key])
    man_path = os.path.join(res_dir, 'arm_manifest.json')
    pv_path = os.path.join(res_dir, 'pool_variability.json')
    if os.path.exists(pv_path):
        pv = json.load(open(pv_path))
        print(f"{TAGS[key]}: fuel single-groups {pv['share_single_completion']:.1%}, "
              f"mean tokens {pv['mean_completion_tokens']:.1f}")
    if os.path.exists(man_path):
        attempts = json.load(open(man_path)).get('attempts', [])
        if sentinel_abort_reason(attempts):
            state = concluded_a2_state(res_dir, attempts)
            print(f'  outcome: a2 sentinel abort, conclusion record {state}')
    sp = os.path.join(res_dir, 'summary.json')
    if not os.path.exists(sp):
        print('  no endpoint summary (not run, aborted, or still training)')
        continue
    s = json.load(open(sp))
    print('  reference       :', s['reference'])
    print('  parent mc-sft   :', s['sft_acc'])
    print('  composition grpo:', s['grpo_acc'])
    print('  grpo delta      :', s['grpo_delta'])
    print('  probe delta     :', s['grpo_logprob_delta'])


029_COLAB_FULL_B15_mcsft_grpo_seed0: fuel single-groups 17.6%, mean tokens 16.0
  outcome: a2 sentinel abort, conclusion record committed
  no endpoint summary (not run, aborted, or still training)
029_COLAB_FULL_B15_mcsft_grpo_seed1: fuel single-groups 26.9%, mean tokens 16.3
  reference       : mcsft_parent:010_p8_COLAB_FULL_B15_sft_only_seed1
  parent mc-sft   : {'easy': 0.7997724687144482, 'medium': 0.6880131362889984, 'hard': 0.45202020202020204}
  composition grpo: {'easy': 0.8031854379977247, 'medium': 0.6880131362889984, 'hard': 0.44696969696969696}
  grpo delta      : {'medium': 0.0, 'hard': -0.005050505050505083, 'easy': 0.0034129692832765013}
  probe delta     : {'medium': -0.0016420361247947435, 'hard': -0.0050505050505050275, 'easy': -0.0011376564277587597}
029_COLAB_FULL_B15_mcsft_grpo_seed2: fuel single-groups 40.8%, mean tokens 16.7
  outcome: a2 sentinel abort, conclusion record committed
  no endpoint summary (not run, aborted, or still training)
029_COLAB_FULL_B3_mcs

## 14. Close the units record

In [18]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 654.13      # fill from the resources panel
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

logged: {'session_id': '20260812_082351', 'ts_start': '2026-08-12 08:24', 'notebook': '029_colab_mcsft_grpo_small', 'colab_cu_before': 671.94, 'unit_rate': 6.77, 'ts_end': '2026-08-12 11:02', 'colab_cu_after': 654.13, 'colab_cu_used': 17.81, 'closed_on_resume': False}
